In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
import joblib

In [2]:
chunks = pd.read_csv('../data/accepted_2007_to_2018Q4.csv', chunksize=100000, parse_dates=['issue_d'], low_memory=False)
df_filtered = pd.concat([c[(c['issue_d'] >= '2013-01-01') & (c['issue_d'] <= '2018-12-31')] for c in chunks])

/var/folders/3f/9mw8q0xn58v4947ln1hhc4mr0000gn/T/ipykernel_3152/2014599947.py:2: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_filtered = pd.concat([c[(c['issue_d'] >= '2013-01-01') & (c['issue_d'] <= '2018-12-31')] for c in chunks])
/var/folders/3f/9mw8q0xn58v4947ln1hhc4mr0000gn/T/ipykernel_3152/2014599947.py:2: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_filtered = pd.concat([c[(c['issue_d'] >= '2013-01-01') & (c['issue_d'] <= '2018-12-31')] for c in chunks])
/var/folders/3f/9mw8q0xn58v4947ln1hhc4mr0000gn/T/ipykernel_3152/2014599947.py:2: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please

In [3]:
df_filtered[0:1].to_dict()

{'id': {0: 68407277},
 'member_id': {0: nan},
 'loan_amnt': {0: 3600.0},
 'funded_amnt': {0: 3600.0},
 'funded_amnt_inv': {0: 3600.0},
 'term': {0: ' 36 months'},
 'int_rate': {0: 13.99},
 'installment': {0: 123.03},
 'grade': {0: 'C'},
 'sub_grade': {0: 'C4'},
 'emp_title': {0: 'leadman'},
 'emp_length': {0: '10+ years'},
 'home_ownership': {0: 'MORTGAGE'},
 'annual_inc': {0: 55000.0},
 'verification_status': {0: 'Not Verified'},
 'issue_d': {0: Timestamp('2015-12-01 00:00:00')},
 'loan_status': {0: 'Fully Paid'},
 'pymnt_plan': {0: 'n'},
 'url': {0: 'https://lendingclub.com/browse/loanDetail.action?loan_id=68407277'},
 'desc': {0: nan},
 'purpose': {0: 'debt_consolidation'},
 'title': {0: 'Debt consolidation'},
 'zip_code': {0: '190xx'},
 'addr_state': {0: 'PA'},
 'dti': {0: 5.91},
 'delinq_2yrs': {0: 0.0},
 'earliest_cr_line': {0: 'Aug-2003'},
 'fico_range_low': {0: 675.0},
 'fico_range_high': {0: 679.0},
 'inq_last_6mths': {0: 1.0},
 'mths_since_last_delinq': {0: 30.0},
 'mths_sinc

In [4]:
df_filtered['loan_status'].value_counts()

loan_status
Fully Paid            997912
Current               878317
Charged Off           254245
Late (31-120 days)     21467
In Grace Period         8436
Late (16-30 days)       4349
Default                   40
Name: count, dtype: int64

In [5]:
df_filtered['term'].value_counts()

term
36 months    1534750
60 months     630016
Name: count, dtype: int64

In [6]:
cat_cols = ['term', 'grade', 'sub_grade', 'emp_length', 'home_ownership'
            , 'verification_status', 'loan_status', 'purpose', 'addr_state'
            , 'initial_list_status', 'application_type']

#### Testing for Multicolinearity

due to the the size of variables and possible correlation between them , it is essential to test for this and reduce the size of variables as necessary.
since an explainable model is desired, PCA will not be used at this point.

In [7]:
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.model_selection import cross_val_score

In [8]:
df = df_filtered.copy()

In [9]:
cols = df_filtered.columns.to_list()

In [10]:
def col_to_num(df, amb_cols):
    for c in amb_cols:
        if c in ['emp_length', 'term']:
            df[c] = df[c].str.extract('(\d+)').astype('float64')
        df[c] = df[c].astype('float64')
    return df

In [11]:
amb_cols = ['emp_length', 'term']
df2 = col_to_num(df, amb_cols)
df2['emp_length'].dtype

dtype('float64')

In [12]:
df_filtered['emp_length'].value_counts()

emp_length
10+ years    723240
2 years      194028
< 1 year     180786
3 years      172517
1 year       141157
5 years      131697
4 years      128994
6 years       96510
8 years       88037
7 years       87783
9 years       76147
Name: count, dtype: int64

In [13]:
keep_cols = ['funded_amnt', 'term','int_rate','installment', 'emp_length','home_ownership',
    'annual_inc','verification_status','pymnt_plan','title','purpose', 'zip_code','addr_state','dti',
    'delinq_2yrs','inq_last_6mths','mths_since_last_delinq','mths_since_last_record','pub_rec','revol_bal',
    'revol_util','total_acc','initial_list_status','out_prncp_inv','policy_code','application_type',
    'annual_inc_joint','dti_joint','verification_status_joint','total_rev_hi_lim','inq_fi',
    'total_cu_tl','inq_last_12m','acc_open_past_24mths','avg_cur_bal','bc_open_to_buy','bc_util',
    'mo_sin_old_il_acct','mo_sin_old_rev_tl_op','mo_sin_rcnt_rev_tl_op',
    'mo_sin_rcnt_tl','mort_acc','mths_since_recent_bc','mths_since_recent_inq','num_actv_bc_tl','num_actv_rev_tl',
    'num_bc_sats','num_bc_tl','num_il_tl','num_op_rev_tl','num_rev_accts','num_rev_tl_bal_gt_0',
    'num_sats','num_tl_op_past_12m','percent_bc_gt_75','pub_rec_bankruptcies','tax_liens',
    'tot_hi_cred_lim','total_bal_ex_mort','total_bc_limit','total_il_high_credit_limit',
    'revol_bal_joint','sec_app_inq_last_6mths','sec_app_mort_acc','sec_app_open_acc',
    'sec_app_revol_util','sec_app_open_act_il','sec_app_num_rev_accts',
    'loan_status','grade','sub_grade', 'issue_d', 'fico_range_low', 'fico_range_high', 
    'last_fico_range_high', 'last_fico_range_low',
]

In [14]:
df_filtered = df_filtered[keep_cols]

In [15]:
# df_filtered = df_filtered['emp_length' ].astype('float64')

In [16]:
cat_cols = df_filtered.select_dtypes(include=['object']).columns
cat_cols

Index(['term', 'emp_length', 'home_ownership', 'verification_status',
       'pymnt_plan', 'title', 'purpose', 'zip_code', 'addr_state',
       'initial_list_status', 'application_type', 'verification_status_joint',
       'loan_status', 'grade', 'sub_grade'],
      dtype='object')

In [17]:
num_cols = [i for i in df_filtered.columns if i not in cat_cols]
df_num = df_filtered[num_cols]

In [18]:
# plt.figure(figsize = (18, 18))
# heatmap = sns.heatmap(df_num.corr(), vmin = -1, vmax = 1, annot = True)
# heatmap.set_title('Numerical Features Correlation Heatmap',  pad = 12)

In [19]:
len(num_cols)

61

In [20]:
cor_mat = df_num.corr()
cor_mat

,funded_amnt,int_rate,installment,annual_inc,dti,delinq_2yrs,inq_last_6mths,mths_since_last_delinq,mths_since_last_record,pub_rec,...,sec_app_mort_acc,sec_app_open_acc,sec_app_revol_util,sec_app_open_act_il,sec_app_num_rev_accts,issue_d,fico_range_low,fico_range_high,last_fico_range_high,last_fico_range_low
funded_amnt,1.000000,0.089843,0.945399,0.195100,0.039782,-0.011488,-0.023259,-0.010902,0.014632,-0.062819,...,0.190662,0.214282,0.029076,0.057999,0.198691,0.030626,0.115286,0.115285,0.095102,0.078760
int_rate,0.089843,1.000000,0.116460,-0.052870,0.123828,0.056589,0.192952,-0.043485,-0.014599,0.053449,...,-0.121385,-0.027289,0.251526,0.026256,-0.074007,-0.059301,-0.405293,-0.405288,-0.351902,-0.275548
installment,0.945399,0.116460,1.000000,0.188075,0.041296,-0.000035,0.003034,-0.020483,-0.000441,-0.050432,...,0.142424,0.191848,0.060896,0.046906,0.177223,0.018712,0.058511,0.058510,0.063172,0.053981
annual_inc,0.195100,-0.052870,0.188075,1.000000,-0.082896,0.025404,0.021129,-0.030535,-0.056971,-0.003389,...,0.077639,0.091330,0.006987,0.018759,0.095166,0.018163,0.037369,0.037369,0.035780,0.033330
dti,0.039782,0.123828,0.041296,-0.082896,1.000000,-0.014136,-0.007590,0.013913,0.058815,-0.029220,...,0.058284,0.041367,0.033873,0.043261,0.017707,0.040110,-0.020830,-0.020833,-0.032820,-0.018536
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
issue_d,0.030626,-0.059301,0.018712,0.018163,0.040110,-0.029119,-0.118280,0.040813,0.110406,-0.032973,...,-0.003665,-0.001164,-0.064562,-0.001304,-0.009037,1.000000,0.139920,0.139919,0.150412,0.129014
fico_range_low,0.115286,-0.405293,0.058511,0.037369,-0.020830,-0.177676,-0.100595,0.101693,0.257356,-0.189545,...,0.052096,0.043793,-0.301463,0.028052,0.059885,0.139920,1.000000,1.000000,0.401236,0.293703
fico_range_high,0.115285,-0.405288,0.058510,0.037369,-0.020833,-0.177673,-0.100596,0.101692,0.257356,-0.189542,...,0.052087,0.043792,-0.301444,0.028049,0.059884,0.139919,1.000000,1.000000,0.401236,0.293702
last_fico_range_high,0.095102,-0.351902,0.063172,0.035780,-0.032820,-0.097238,-0.125417,0.077455,0.088392,-0.077514,...,0.078083,0.029384,-0.185563,0.003881,0.053753,0.150412,0.401236,0.401236,1.000000,0.845655


In [21]:
# cor_mat['open_acc'].sort_values(ascending=False).to_dict()

In [22]:

corr_matrix = cor_mat.abs()   

strong_corr = {
    col: {
        other_col: round(float(corr_matrix.loc[col, other_col]), 4)
        for other_col in corr_matrix.columns
        if other_col != col and corr_matrix.loc[col, other_col] > 0.75
    }
    for col in corr_matrix.columns
}

# Print each column and the other columns with their correlation values
for col, vals in strong_corr.items():
    if vals:
        print(f"{col}: {vals}")

funded_amnt: {'installment': 0.9454}
installment: {'funded_amnt': 0.9454}
annual_inc: {'annual_inc_joint': 0.7816}
revol_bal: {'total_rev_hi_lim': 0.8044}
revol_util: {'bc_util': 0.8685}
total_acc: {'num_rev_accts': 0.7646}
annual_inc_joint: {'annual_inc': 0.7816}
total_rev_hi_lim: {'revol_bal': 0.8044, 'total_bc_limit': 0.7555}
acc_open_past_24mths: {'num_tl_op_past_12m': 0.765}
avg_cur_bal: {'tot_hi_cred_lim': 0.7872}
bc_open_to_buy: {'total_bc_limit': 0.8436}
bc_util: {'revol_util': 0.8685, 'percent_bc_gt_75': 0.8465}
num_actv_bc_tl: {'num_actv_rev_tl': 0.8212, 'num_bc_sats': 0.8378, 'num_rev_tl_bal_gt_0': 0.8145}
num_actv_rev_tl: {'num_actv_bc_tl': 0.8212, 'num_op_rev_tl': 0.7988, 'num_rev_tl_bal_gt_0': 0.9835}
num_bc_sats: {'num_actv_bc_tl': 0.8378, 'num_op_rev_tl': 0.7625}
num_bc_tl: {'num_rev_accts': 0.8399}
num_op_rev_tl: {'num_actv_rev_tl': 0.7988, 'num_bc_sats': 0.7625, 'num_rev_accts': 0.7948, 'num_rev_tl_bal_gt_0': 0.8029, 'num_sats': 0.837}
num_rev_accts: {'total_acc': 0.7

In [23]:
drop_cols=['num_rev_tl_bal_gt_0']

In [24]:
features = df_filtered.drop(columns=drop_cols, axis=1 )

In [25]:
cat_cols

Index(['term', 'emp_length', 'home_ownership', 'verification_status',
       'pymnt_plan', 'title', 'purpose', 'zip_code', 'addr_state',
       'initial_list_status', 'application_type', 'verification_status_joint',
       'loan_status', 'grade', 'sub_grade'],
      dtype='object')

In [26]:
loan_reasons = df_filtered['purpose'].unique().tolist()
loan_reasons

['debt_consolidation',
 'small_business',
 'home_improvement',
 'major_purchase',
 'credit_card',
 'other',
 'house',
 'vacation',
 'car',
 'medical',
 'moving',
 'renewable_energy',
 'wedding',
 'educational']

In [ ]:
# df = pd.DataFrame({"loan_reason": loan_reasons})

# def clean_reason(x):
#     x = str(x).lower().strip()
#     x = x.replace("&", " and ")
#     x = re.sub(r"[^a-z0-9\s]", " ", x)   
#     x = re.sub(r"\s+", " ", x)            
#     return x.strip()



In [ ]:
import re
# df["loan_reason_clean"] = df["loan_reason"].map(clean_reason)

# # aggregated counts
# agg = (
#     df.groupby("loan_reason_clean")
#       .size()
#       .reset_index(name="count")
#       .sort_values("count", ascending=False)
# )



In [ ]:
# agg.tail(500)

,loan_reason_clean,count
9017,dental braces,1
9016,dental bill repayment,1
9015,dental bill,1
9014,dental 101,1
9011,demolish credit card debt,1
...,...,...
9482,edu,1
9481,economy shuffle,1
9480,economy car,1
9479,economic freedom,1


In [ ]:
# def classify_loan_reason(x):
#     text = clean_reason(x)
#     # credit card / refinance / cc payoff
#     if re.search(r"(credit card|creditcard|cc|refinance|refi|refinancing|payoff|pay off)", text):
#         return "credit_card_refinancing"

#     # car financing
#     if re.search(r"(car|auto|vehicle|automobile|car loan|car refinance)", text):
#         return "car_financing"

#     # home improvement
#     if re.search(r"(home improvement|home improv|repair|remodel|renovation|roof|kitchen|bath|home)", text):
#         return "home_improvement"

#     # medical expenses
#     if re.search(r"(medical|dental|doctor|hospital|surgery|health|vision|clinic|emergency)", text):
#         return "medical_expenses"

#     # debt consolidation
#     if re.search(r"(consol|consolidat|debt|consolidate)", text):
#         return "debt_consolidation"
#     # Optional fallback
#     return text

In [ ]:
# features['loan_reason_category'] = df_filtered['title'].map(clean_reason).map(classify_loan_reason)

In [ ]:
# group_counts = features['loan_reason_category'].value_counts()
# keep = group_counts[group_counts > 1000].index.tolist()
# features['loan_reason_category'] = features['loan_reason_category'].apply(lambda x: x if x in keep else 'other')

In [ ]:
# keep

['debt_consolidation',
 'credit_card_refinancing',
 'home_improvement',
 'other',
 'major purchase',
 'medical_expenses',
 'nan',
 'car_financing',
 'business',
 'vacation',
 'moving and relocation',
 'personal loan',
 'green loan',
 'personal']

In [ ]:
# features['loan_reason_category'].unique()

array(['debt_consolidation', 'business', 'nan', 'major purchase',
       'credit_card_refinancing', 'home_improvement', 'other', 'vacation',
       'car_financing', 'medical_expenses', 'moving and relocation',
       'green loan', 'personal', 'personal loan'], dtype=object)

In [27]:
features = df_filtered.copy()

In [28]:
features['issue_d'] = pd.to_datetime(features['issue_d'], errors='coerce')

date_cols = [
    'hardship_start_date',
    'hardship_end_date',
    'last_pymnt_d',
    'next_pymnt_d',
    'last_credit_pull_d',
    'debt_settlement_flag_date',
    'settlement_date',
    'payment_plan_start_date',
    'earliest_cr_line',
    'sec_app_earliest_cr_line'
]

for col in date_cols:
    if col in features.columns:
        features[col] = pd.to_datetime(features[col], errors='coerce')
        features[f'{col}_days_from_issue'] = (
            (features[col] - features['issue_d']).dt.days
        )

In [30]:
# features = features.drop(columns = date_cols, axis=1 )

In [35]:
features['issue_d']

0         2015-12-01
1         2015-12-01
2         2015-12-01
3         2015-12-01
4         2015-12-01
             ...    
2260694   2016-10-01
2260695   2016-10-01
2260696   2016-10-01
2260697   2016-10-01
2260698   2016-10-01
Name: issue_d, Length: 2164766, dtype: datetime64[ns]

In [32]:
features['loan_cohort'] = features['issue_d'].dt.to_period('M').astype(str)

In [33]:
features.head()

,funded_amnt,term,int_rate,installment,emp_length,home_ownership,annual_inc,verification_status,pymnt_plan,title,...,sec_app_num_rev_accts,loan_status,grade,sub_grade,issue_d,fico_range_low,fico_range_high,last_fico_range_high,last_fico_range_low,loan_cohort
0,3600.0,36 months,13.99,123.03,10+ years,MORTGAGE,55000.0,Not Verified,n,Debt consolidation,...,NaN,Fully Paid,C,C4,2015-12-01,675.0,679.0,564.0,560.0,2015-12
1,24700.0,36 months,11.99,820.28,10+ years,MORTGAGE,65000.0,Not Verified,n,Business,...,NaN,Fully Paid,C,C1,2015-12-01,715.0,719.0,699.0,695.0,2015-12
2,20000.0,60 months,10.78,432.66,10+ years,MORTGAGE,63000.0,Not Verified,n,NaN,...,NaN,Fully Paid,B,B4,2015-12-01,695.0,699.0,704.0,700.0,2015-12
3,35000.0,60 months,14.85,829.90,10+ years,MORTGAGE,110000.0,Source Verified,n,Debt consolidation,...,NaN,Current,C,C5,2015-12-01,785.0,789.0,679.0,675.0,2015-12
4,10400.0,60 months,22.45,289.91,3 years,MORTGAGE,104433.0,Source Verified,n,Major purchase,...,NaN,Fully Paid,F,F1,2015-12-01,695.0,699.0,704.0,700.0,2015-12


In [34]:
df_filtered.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2164766 entries, 0 to 2260698
Data columns (total 76 columns):
 #   Column                      Dtype         
---  ------                      -----         
 0   funded_amnt                 float64       
 1   term                        object        
 2   int_rate                    float64       
 3   installment                 float64       
 4   emp_length                  object        
 5   home_ownership              object        
 6   annual_inc                  float64       
 7   verification_status         object        
 8   pymnt_plan                  object        
 9   title                       object        
 10  purpose                     object        
 11  zip_code                    object        
 12  addr_state                  object        
 13  dti                         float64       
 14  delinq_2yrs                 float64       
 15  inq_last_6mths              float64       
 16  mths_since_last_delinq 

In [35]:
features = col_to_num(features, amb_cols)

In [36]:
cat_cols

Index(['term', 'emp_length', 'home_ownership', 'verification_status',
       'pymnt_plan', 'title', 'purpose', 'zip_code', 'addr_state',
       'initial_list_status', 'application_type', 'verification_status_joint',
       'loan_status', 'grade', 'sub_grade'],
      dtype='object')

In [37]:
features['loan_status'].value_counts()

loan_status
Fully Paid            997912
Current               878317
Charged Off           254245
Late (31-120 days)     21467
In Grace Period         8436
Late (16-30 days)       4349
Default                   40
Name: count, dtype: int64

In [38]:
def targets(x):
    if x in ['Fully Paid']:
        return 0
    elif x in ['Charged Off', 'Default', 'Late (31-120 days)']:
        return 1
    else:
        return 2

In [39]:
features['targets'] = features['loan_status'].map(targets)

In [40]:
cat_cols = features.select_dtypes(include=['object']).columns

In [41]:
cat_cols

Index(['home_ownership', 'verification_status', 'pymnt_plan', 'title',
       'purpose', 'zip_code', 'addr_state', 'initial_list_status',
       'application_type', 'verification_status_joint', 'loan_status', 'grade',
       'sub_grade', 'loan_cohort'],
      dtype='object')

In [42]:
# features['pymnt_plan'] = features['pymnt_plan'].map({'n': 0, 'y': 1})
# features['hardship_flag'] = features['hardship_flag'].map({'N': 0, 'Y': 1})

In [43]:
features['verification_status_joint'] = features['verification_status_joint'].map({'Not Verified': 0, 'Source Verified': 1, 'Verified': 2})


In [44]:
# status_map = {
#     "Current": 0,
#     "Issued": 0,
#     "In Grace Period": 1,
#     "Late (16-30 days)": 1,
#     "Late (31-120 days)": 2,
# }

# features["hardship_loan_status"] = (
#     features["hardship_loan_status"]
#     .map(status_map)
#     .fillna(features["hardship_loan_status"]) 
# )

In [57]:
drop2 = ['title', 'issue_d']

In [58]:
# features['hardship_loan_status'].value_counts()

In [59]:
data = features.drop(columns=drop2, axis=1)

In [60]:
data = data[data['targets'] != 2]

In [61]:
cat_labels = data.select_dtypes(include=['object']).columns

In [62]:
cat_labels

Index(['home_ownership', 'verification_status', 'pymnt_plan', 'purpose',
       'zip_code', 'addr_state', 'initial_list_status', 'application_type',
       'loan_status', 'grade', 'sub_grade', 'loan_cohort'],
      dtype='object')

In [76]:
non_features = ['grade', 'sub_grade', 'loan_status', 'fico_range_low', 'loan_cohort',
               'fico_range_low','fico_range_high', 'last_fico_range_high', 'last_fico_range_low', 'targets']

Model Data Prep

In [72]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder

scaler = StandardScaler()
label_encoder = LabelEncoder()


In [73]:
def scaler_and_encode(df, cat_labels, non_features):
    # Scale numerical features
    num_cols = df.select_dtypes(include=['float64', 'int64']).columns
    for col in num_cols:
        if col not in non_features:
            df[col] = scaler.fit_transform(df[[col]])

    # Encode categorical features
    for col in cat_labels:
        if col not in non_features:  # Exclude 'targets' from encoding
            df[col] = label_encoder.fit_transform(df[col].astype(str))
    if col == 'targets':  # Exclude 'issue_d' from encoding
            df[col] = label_encoder.fit_transform(df[col].astype(str))

    return df

In [74]:
data_scaled = scaler_and_encode(data.copy(), cat_labels, non_features)

In [77]:
X = data_scaled.drop(columns=non_features, axis=1)
y = data_scaled['targets']

In [78]:
X.columns

Index(['funded_amnt', 'term', 'int_rate', 'installment', 'emp_length',
       'home_ownership', 'annual_inc', 'verification_status', 'pymnt_plan',
       'purpose', 'zip_code', 'addr_state', 'dti', 'delinq_2yrs',
       'inq_last_6mths', 'mths_since_last_delinq', 'mths_since_last_record',
       'pub_rec', 'revol_bal', 'revol_util', 'total_acc',
       'initial_list_status', 'out_prncp_inv', 'policy_code',
       'application_type', 'annual_inc_joint', 'dti_joint',
       'verification_status_joint', 'total_rev_hi_lim', 'inq_fi',
       'total_cu_tl', 'inq_last_12m', 'acc_open_past_24mths', 'avg_cur_bal',
       'bc_open_to_buy', 'bc_util', 'mo_sin_old_il_acct',
       'mo_sin_old_rev_tl_op', 'mo_sin_rcnt_rev_tl_op', 'mo_sin_rcnt_tl',
       'mort_acc', 'mths_since_recent_bc', 'mths_since_recent_inq',
       'num_actv_bc_tl', 'num_actv_rev_tl', 'num_bc_sats', 'num_bc_tl',
       'num_il_tl', 'num_op_rev_tl', 'num_rev_accts', 'num_rev_tl_bal_gt_0',
       'num_sats', 'num_tl_op_past_12m

In [79]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

Logistic Regression Model

In [80]:
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
logreg = LogisticRegression(max_iter=100000, random_state=42)


In [81]:
num_cols  = [ c for c in X_train.columns if c not in cat_labels]

In [82]:
cats = [c for c in cat_labels if c not in non_features]

In [83]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler())
            ]),
            num_cols
        ),
        (
            "cat",
            Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent"))
                # ("onehot", LabelEncoder(handle_unknown="ignore"))
            ]),
            cats
        )
    ]
)

In [85]:
data_sample = data_scaled.sample(n=500000, random_state=42)
X_sample = data_sample.drop(columns=non_features, axis=1)
y_sample = data_sample['targets']

In [86]:
X_sample_train, X_sample_test, y_sample_train, y_sample_test = train_test_split(X_sample, y_sample, test_size=0.3, random_state=42)

In [87]:
model = Pipeline([
    ("preprocessor", preprocessor),
    ("scaler", StandardScaler()),
    ("logreg", LogisticRegression(max_iter=5000, solver="lbfgs", random_state=42))
])

cv_scores = cross_val_score(model, X_sample_train, y_sample_train, cv=cv, scoring="accuracy")


In [88]:
print("CV scores:", cv_scores)

CV scores: [0.8021     0.80152857 0.80288571 0.80231429 0.80415714]


In [89]:
sample_model = Pipeline([
    ("preprocessor", preprocessor),
    ("scaler", StandardScaler()),
    ("logreg", LogisticRegression(max_iter=5000, solver="lbfgs", random_state=42))
])

In [90]:
sample_model.fit(X_sample_train, y_sample_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('scaler', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transform

In [91]:
pred = sample_model.predict(X_sample_test)


In [92]:
from sklearn.metrics import classification_report,precision_score,recall_score,f1_score,roc_auc_score,accuracy_score


In [93]:
print("Classification Report:\n", classification_report(y_sample_test, pred))

Classification Report:
               precision    recall  f1-score   support

           0       0.81      0.98      0.89    117693
           1       0.69      0.17      0.27     32307

    accuracy                           0.80    150000
   macro avg       0.75      0.57      0.58    150000
weighted avg       0.78      0.80      0.75    150000



In [106]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

model = Pipeline([
    ("preprocessor", preprocessor),
    ("scaler", StandardScaler()),
    ("logreg", LogisticRegression(max_iter=5000,class_weight="balanced",random_state=42))
])


In [107]:
model.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('scaler', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transform

In [108]:

y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.88      0.70      0.78    199583
           1       0.37      0.64      0.47     55150

    accuracy                           0.69    254733
   macro avg       0.62      0.67      0.63    254733
weighted avg       0.77      0.69      0.71    254733



In [ ]:
# model.fit(X_train, y_train)

In [109]:
from sklearn.ensemble import RandomForestClassifier


In [119]:
from sklearn.utils.class_weight import compute_class_weight
classes = np.unique(y_train)
weights = compute_class_weight(class_weight="balanced",classes=classes,y=y_train)

In [120]:
class_weight_dict = dict(zip(classes, weights))

In [121]:
rfmodel = Pipeline([
    ("preprocessor", preprocessor),
    ("scaler", StandardScaler()),
    ("rf", RandomForestClassifier(n_estimators=300,max_depth=None,min_samples_leaf=10,
    random_state=42,class_weight=class_weight_dict))])

In [122]:
rfmodel.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('scaler', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transform

In [112]:
rf_pred = rfmodel.predict(X_test)

In [113]:
print("Classification Report:\n", classification_report(y_test, rf_pred))

Classification Report:
               precision    recall  f1-score   support

           0       0.85      0.87      0.86    199583
           1       0.48      0.44      0.46     55150

    accuracy                           0.78    254733
   macro avg       0.66      0.65      0.66    254733
weighted avg       0.77      0.78      0.77    254733



In [103]:
test_pred = rfmodel.predict(X_train)
print("Classification Report:\n", classification_report(y_train, test_pred))


Classification Report:
               precision    recall  f1-score   support

           0       0.82      0.99      0.90    698253
           1       0.83      0.23      0.35    193311

    accuracy                           0.82    891564
   macro avg       0.82      0.61      0.63    891564
weighted avg       0.82      0.82      0.78    891564



In [118]:
y.sum()/y.count()

np.float64(0.21650293955077635)

In [ ]:
#saving model
joblib.dump(rfmodel, '../api/models/rfmodel.pkl')

['../api/models/rfmodel.pkl']

In [ ]:
data_2018 = data_scaled[data_scaled['issue_d'].dt.year == 2018]

In [ ]:
# full_test_pred = rfmodel.predict(X_train)
# print("Classification Report:\n", classification_report(y_train, full_test_pred))

In [86]:
train_cols = X_train.columns
print(train_cols)

Index(['funded_amnt', 'term', 'int_rate', 'installment', 'emp_length',
       'home_ownership', 'annual_inc', 'verification_status', 'pymnt_plan',
       'purpose',
       ...
       'hardship_end_date_days_from_issue', 'last_pymnt_d_days_from_issue',
       'next_pymnt_d_days_from_issue', 'last_credit_pull_d_days_from_issue',
       'debt_settlement_flag_date_days_from_issue',
       'settlement_date_days_from_issue',
       'payment_plan_start_date_days_from_issue',
       'earliest_cr_line_days_from_issue',
       'sec_app_earliest_cr_line_days_from_issue', 'loan_cohort'],
      dtype='object', length=128)


In [96]:
train_cols[0:50]

Index(['funded_amnt', 'term', 'int_rate', 'installment', 'emp_length',
       'home_ownership', 'annual_inc', 'verification_status', 'pymnt_plan',
       'purpose', 'zip_code', 'addr_state', 'dti', 'delinq_2yrs',
       'inq_last_6mths', 'mths_since_last_delinq', 'mths_since_last_record',
       'pub_rec', 'revol_bal', 'revol_util', 'total_acc',
       'initial_list_status', 'out_prncp_inv', 'total_pymnt_inv',
       'total_rec_prncp', 'total_rec_int', 'total_rec_late_fee', 'recoveries',
       'last_pymnt_amnt', 'collections_12_mths_ex_med',
       'mths_since_last_major_derog', 'policy_code', 'application_type',
       'annual_inc_joint', 'dti_joint', 'verification_status_joint',
       'acc_now_delinq', 'tot_coll_amt', 'tot_cur_bal', 'open_acc_6m',
       'open_act_il', 'open_il_12m', 'open_il_24m', 'mths_since_rcnt_il',
       'total_bal_il', 'il_util', 'open_rv_12m', 'open_rv_24m', 'max_bal_bc',
       'all_util'],
      dtype='object')

In [94]:
df_filtered['settlement_status'].value_counts()

settlement_status
ACTIVE      14686
COMPLETE    14045
BROKEN       4969
Name: count, dtype: int64